# Strip GNN-benchmark skipped UniProt IDs from 3D-CNN bench CSVs

**Source log**: `Catalytic_site_prediction_setA_gnn/logs/catGNN_bench_272998.out` — lines with `[SKIP] <ID>: max_std_resid=... > ESM_len=1021` (and PDB parse skips).

**Input**: per-residue `bench_enzyme_<stamp>.csv` / `bench_nonenzyme_<stamp>.csv` (columns: `accession`, `res_num`, `res_type`, `prob`).

**Output**: `bench_enzyme_<stamp>_delete.csv` (rows whose `accession` is in the enzyme skip list removed). Use the non-enzyme skip list the same way for `bench_nonenzyme_*.csv`.

**Note**: This matches the same proteins the GNN benchmark did not score; removing them keeps enzyme vs non-enzyme comparisons aligned with the GNN run.

If your `bench_enzyme_*.csv` is a **small debug subset**, none of the skip IDs may appear — the script still writes `_delete.csv` with **zero rows removed** (and prints `[info] skip list entries not present in csv`). Full benchmark outputs list ~942 enzymes.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

# catGNN_bench_272998.out — [SKIP] lines under "=== Inference: ENZYME set ==="
SKIP_ENZYME = frozenset(
    [
        "A0Q7Q2",
        "C9X1G5",
        "H9T8G6",
        "J3F2B0",
        "J7RUA5",
        "P0CQ11",
        "Q01804",
        "Q03LF7",
        "Q07794",
        "Q6NKI3",
        "Q6ZPJ3",
        "Q927P4",
        "Q9D8C3",
        "Q9ZVX1",
        "U2UMQ6",
    ]
)

# Same log — [SKIP] lines under "=== Inference: NON-ENZYME set ===" (for bench_nonenzyme_*.csv)
SKIP_NON_ENZYME = frozenset(
    [
        "Q9VAW5",
        "Q8N7Z5",
        "Q2UKX8",
        "O75155",
        "P38164",
        "P0CR87",
        "Q1EAK2",
        "O15399",
        "Q9PL46",
        "P0C6C0",
        "P06593",
        "Q9QWK5",
        "P15001",
        "E1C2U2",
        "Q9Y238",
        "Q8BI84",
        "Q5RG44",
        "Q8MLZ5",
        "A1A5F2",
        "Q9ULI4",
        "Q0JBY9",
        "P18614",
        "F4HRT5",
        "Q8BM75",
        "O43896",
        "Q9ESD7",
        "Q62470",
        "O14924",
        "Q8LLD0",
        "Q869Q3",
        "Q86BY9",
        "P08953",
        "Q9P2H0",
        "P32492",
        "Q9W517",
        "Q5RHB5",
        "B5YUD4",
        "Q4WWL0",
        "Q54GS1",
        "Q9Y4D1",
        "Q2LKV5",
        "Q9LZX8",
        "Q9P265",
        "O84818",
        "Q2PFD7",
        "Q505K2",
        "Q980C7",
        "Q67C55",
        "Q5XK85",
        "Q75JP5",
        "Q5SW79",
        "B7LE23",
        "Q6C5F1",
        "Q3U3I9",
        "Q9P2E7",
        "Q8ZG99",
        "Q71M21",
        "P32257",
        "Q74Z37",
        "Q79FI9",
        "Q12770",
        "Q54L90",
        "Q6DFF2",
        "P38894",
        "Q9NGP5",
        "P98163",
        "B0E2U2",
        "Q10NY2",
        "P54289",
        "E9PT23",
        "Q2VQM6",
        "P32521",
        "P54816",
        "Q5RJF7",
        "Q9HCD6",
        "P11137",
        "Q9Y6C5",
        "B3LR49",
        "Q5RHH4",
        "Q8WYP5",
        "P28166",
        "O94913",
    ]
)

BENCH_ENZYME_CSV = Path(
    "/data/data3/conglab/s441865/code/Catalytic_sites_setA_3dcnn/results/run_seed42/bench_enzyme_20260429_201232.csv"
)

In [ ]:
def filter_bench_csv(src: Path, skip_accessions: frozenset, *, kind: str) -> Path:
    """Drop all rows whose `accession` is in skip_accessions; write `<stem>_delete.csv` beside src."""
    if not src.is_file():
        raise FileNotFoundError(src)
    dst = src.with_name("{}_delete{}".format(src.stem, src.suffix))
    df = pd.read_csv(src)
    if "accession" not in df.columns:
        raise ValueError("Expected column 'accession' in {}".format(src))
    mask = df["accession"].isin(skip_accessions)
    n_drop = int(mask.sum())
    kept = df.loc[~mask].copy()
    kept.to_csv(dst, index=False)
    print(
        "[{}] {}  rows_in={}  rows_out={}  dropped_accessions={}  -> {}".format(
            kind,
            src.name,
            len(df),
            len(kept),
            n_drop,
            dst,
        )
    )
    dropped_ids = sorted(df.loc[mask, "accession"].unique().tolist())
    extra = sorted(skip_accessions - set(dropped_ids))
    missing = sorted(set(dropped_ids) - skip_accessions)
    if extra:
        print("  [info] skip list entries not present in csv:", extra)
    if missing:
        print("  [warn] dropped rows with accession not in skip list?", missing)
    return dst


_ = filter_bench_csv(BENCH_ENZYME_CSV, SKIP_ENZYME, kind="enzyme")

### Optional: `bench_nonenzyme_*.csv`

Uncomment and set the path to filter the non-enzyme bench file with `SKIP_NON_ENZYME`.

In [ ]:
# BENCH_NONENZYME_CSV = Path(
#     "/data/data3/conglab/s441865/code/Catalytic_sites_setA_3dcnn/results/run_seed42/bench_nonenzyme_20260429_201232.csv"
# )
# _ = filter_bench_csv(BENCH_NONENZYME_CSV, SKIP_NON_ENZYME, kind="non-enzyme")